# Imbalance: who gets the concepts?

A demo: a ResNet34 trained from scratch on Imagenette where **one class is ten times rarer**. At every checkpoint,
NAVE's concepts (fitted on a balanced probe set) are assigned to the class they correlate with most, and we
count how many each class gets, next to each class's accuracy. Then the patch view: the shown image's patches in
the UMAP among the others, and its explanation.

**Needs:** `pip install "nave-xai[animation]" torch torchvision`. **Downloads:** Imagenette (about 100 MB) once.

In [ ]:
import sys
if "google.colab" in sys.modules:       # on Google Colab: install what this notebook needs
    get_ipython().run_line_magic("pip", 'install -q "nave-xai[animation] @ git+https://github.com/Ahcene-B/NAVE.git" torch torchvision ipywidgets')
# elsewhere: pip install "nave-xai[animation]" torch torchvision ipywidgets
import numpy as np
import torch

EPOCHS = 10
RARE_CLASS = 0
RARE_SHARE = 0.1
PROBE_PER_CLASS = 20
K = 20
LAYERS = ["layer3", "layer4"]
N_GRAY = 3000                              # patches of other probe images in the UMAP (gray)
UMAP_EPOCHS = 200
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
print("device:", DEVICE)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


def match_colours(labels, reference, K):
    """Give every concept of `labels` the colour index of the reference concept it overlaps most
    (one-to-one, maximising the total overlap), so colours stay stable from frame to frame."""
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels.ravel(), reference.ravel()), 1)
    try:
        from scipy.optimize import linear_sum_assignment
        rows, cols = linear_sum_assignment(-overlap)
        mapping = dict(zip(rows, cols))
    except ImportError:                                     # greedy, without scipy
        mapping, used = {}, set()
        for a, b in sorted(np.ndindex(K, K), key=lambda ab: -overlap[ab]):
            if a not in mapping and b not in used:
                mapping[a] = b
                used.add(b)
    return np.vectorize(lambda a: mapping.get(a, a))(labels)


def knn_agreement(features, targets, k=5, n=1500, seed=0):
    """How well a point's k nearest neighbours (cosine) predict its target: leave-one-out accuracy."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(features), min(n, len(features)), replace=False)
    F = features[idx] / np.linalg.norm(features[idx], axis=1, keepdims=True)
    t = targets[idx]
    sim = F @ F.T
    np.fill_diagonal(sim, -np.inf)
    nearest = np.argsort(-sim, axis=1)[:, :k]
    votes = np.array([np.bincount(t[row], minlength=t.max() + 1).argmax() for row in nearest])
    return float((votes == t).mean())


def normalise_embedding(e):
    e = e - np.median(e, axis=0)
    return e / np.percentile(np.abs(e), 98)


def block_names(backbone):
    """The input of block 1 ("b0"), then every block."""
    return ["b0"] + list(backbone.blocks)


def pool_to(m, gh, gw):
    """Average a (C, h, w) map onto a gh x gw grid (each cell: the mean of the cells it covers)."""
    C, h, w = m.shape
    if (h, w) == (gh, gw):
        return m
    if h % gh == 0 and w % gw == 0:
        return m.reshape(C, gh, h // gh, gw, w // gw).mean((2, 4))
    from skimage.transform import resize
    return resize(m, (C, gh, gw), order=1, anti_aliasing=True, preserve_range=True)


def block_tokens(backbone, images, names):
    """For every block: the tokens of all images on the deepest block's grid, scaled to length 1.
    Returns (tokens per block, (grid height, grid width), cell size in pixels)."""
    mean, std = np.array(backbone.mean, np.float32), np.array(backbone.std, np.float32)
    per_block = {n: [] for n in names}
    grid = None
    for img in images:
        x = ((img / 255.0 - mean) / std).astype(np.float32).transpose(2, 0, 1)[None]
        out = backbone(x, names)
        if grid is None:
            grid = out[names[-1]].shape[2:]
        for n in names:
            t = pool_to(out[n][0], *grid)
            per_block[n].append(t.reshape(t.shape[0], -1).T)
    tokens = [np.concatenate(per_block[n]) for n in names]
    tokens = [t / (np.linalg.norm(t, axis=1, keepdims=True) + 1e-12) for t in tokens]
    return tokens, tuple(grid), images[0].shape[0] // grid[0]


def cell_centres(size, cell):
    return np.arange(cell // 2, size, cell)


def aligned_umap(features, epochs, window=2, regularisation=0.02):
    """UMAP of a sequence of point sets (the same points), each frame kept close to its neighbours."""
    a = umap.AlignedUMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, alignment_regularisation=regularisation,
                         alignment_window_size=window)
    a.fit(features, relations=[{i: i for i in range(len(features[0]))}] * (len(features) - 1))
    return [normalise_embedding(e) for e in a.embeddings_]


def last_reference_umap(features, epochs):
    """UMAP of every frame, each starting from the last frame's layout."""
    last = umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs).fit_transform(features[-1])
    out = [umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, init=last).fit_transform(f) for f in features[:-1]]
    return [normalise_embedding(e) for e in out + [last]]


def renamed_like(labels, reference, K):
    """A renaming of concept ids so each takes the id of the reference concept it overlaps most."""
    from scipy.optimize import linear_sum_assignment
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels, reference), 1)
    r, c = linear_sum_assignment(-overlap)
    rename = np.arange(K)
    rename[r] = c
    return rename

import warnings
import umap

warnings.filterwarnings("ignore", module="umap")               # UMAP's notices about threads and seeds
warnings.filterwarnings("ignore", category=FutureWarning)      # scikit-learn's, inside UMAP

from nave_xai.composite import CompositeFeatures


def to_uint8(x):
    """A (3, H, W) tensor in [0, 1] -> an (H, W, 3) uint8 image, for NAVE."""
    return (x.permute(1, 2, 0).clamp(0, 1).numpy() * 255).astype(np.uint8)


def composite_patches(backbone, images, layers):
    """NAVE's composite features of every patch (on the finest chosen layer's grid) of uint8 images:
    (N, patches, D), the grid and the patch size in pixels."""
    mean, std = np.array(backbone.mean, np.float32), np.array(backbone.std, np.float32)
    out = []
    for im in images:
        x = ((im / 255.0 - mean) / std).astype(np.float32).transpose(2, 0, 1)[None]
        maps = backbone(x, layers)
        f = CompositeFeatures([maps[n][0] for n in layers], "nearest").full()
        out.append(f.reshape(-1, f.shape[-1]))
    return np.stack(out), f.shape[:2], images[0].shape[0] // f.shape[0]


class PatchCloud:
    """The UMAP's points: a fixed sample of patches of the probe images (drawn in gray) and every patch of
    the images that can be shown (drawn in their concepts' colours)."""

    def __init__(self, n_images, n_patches, show, n_gray, seed):
        rng = np.random.default_rng(seed)
        pairs = [(i, g) for i in range(n_images) if i not in show for g in range(n_patches)]
        self.gray = [pairs[p] for p in rng.choice(len(pairs), min(n_gray, len(pairs)), replace=False)]
        self.show, self.G = list(show), n_patches

    def gather(self, feats):
        gray = np.stack([feats[i, g] for i, g in self.gray])
        return np.concatenate([gray, feats[self.show].reshape(-1, feats.shape[-1])])

    def of(self, k):
        """Indices, among the gathered points, of the k-th showable image's patches."""
        start = len(self.gray) + k * self.G
        return np.arange(start, start + self.G)


def colour_snapshots(snaps, K, centres):
    """Match every checkpoint's concepts to the last checkpoint's (per shown image), and colour them."""
    palette = plt.get_cmap("Spectral")(np.linspace(0, 1, K))[:, :3]
    final = snaps[-1]["labels"]
    for s in snaps:
        s["explanation"], s["patch_colours"] = [], []
        for lab, ref in zip(s["labels"], final):
            matched = match_colours(lab, ref, K)
            s["explanation"].append(palette[matched])
            s["patch_colours"].append(palette[matched[np.ix_(centres, centres)].ravel()])
    return palette


PANEL_TITLES = {"input": "image", "umap": "UMAP of patches", "explanation": "NAVE explanation",
                "uncertainty": "uncertainty", "saliency": "NAVE saliency"}


def draw(ax, panel, row, k, image):
    """The content of one panel at checkpoint k, for shown image `image` (index into SHOW)."""
    s = row["snaps"][k]
    if panel == "input":
        return ax.imshow(row["inputs"][image])
    if panel == "explanation":
        return ax.imshow(s["explanation"][image])
    if panel == "uncertainty":
        return ax.imshow(s["uncertainty"][image].astype(float), cmap="inferno_r", vmin=0, vmax=1)
    if panel == "saliency":
        return ax.imshow(s["saliency"][image].astype(float), cmap="seismic", vmin=-1, vmax=1)


def animate_training(rows, curves, image, panels, frames_between=6, title=""):
    """rows: [dict(label, snaps, emb, cloud, inputs)]; curves: [(label, values per checkpoint)];
    panels: any of "input", "umap", "explanation", "uncertainty", "saliency"."""
    snaps0 = rows[0]["snaps"]
    n = len(snaps0)
    plt.rcParams["animation.embed_limit"] = 100
    fig = plt.figure(figsize=(3.0 * len(panels), 3.0 * len(rows) + 1.6), dpi=60)
    grid_ = fig.add_gridspec(len(rows) + 1, len(panels), height_ratios=[3] * len(rows) + [1.1], hspace=0.4)
    arts = []
    for r, row in enumerate(rows):
        img = row.get("image", image)                          # a row may show its own image
        mine = row["cloud"].of(img)
        for j, p in enumerate(panels):
            ax = fig.add_subplot(grid_[r, j]); ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(PANEL_TITLES[p], fontsize=9)
            if j == 0:
                ax.set_ylabel(row["label"], fontsize=9)
            if p == "umap":
                ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
                every = ax.scatter(*row["emb"][0].T, c="0.82", s=3, linewidths=0)
                sel = ax.scatter(*row["emb"][0][mine].T, c=row["snaps"][0]["patch_colours"][img], s=14,
                                 edgecolors="black", linewidths=0.3)
                arts.append((p, row, (every, sel), mine, img))
            else:
                arts.append((p, row, draw(ax, p, row, 0, img), None, img))
    curve = fig.add_subplot(grid_[len(rows), :])
    for label, values in curves:
        curve.plot(range(n), values, marker="o", ms=3, label=label)
    curve.set_xticks(range(n), [str(s["step"]) for s in snaps0], fontsize=6, rotation=45)
    curve.set_ylim(0, 1.02); curve.set_xlabel("training step", fontsize=8); curve.legend(fontsize=7, loc="lower right")
    marker = curve.axvline(0, color="black", lw=1)
    head = fig.suptitle("", fontsize=11)

    def frame(f):
        t = f / frames_between; i0, w = int(t), t - int(t); i1 = min(i0 + 1, n - 1); k = int(round(t))
        for p, row, art, mine, img in arts:
            if p == "umap":
                pos = (1 - w) * row["emb"][i0] + w * row["emb"][i1]
                art[0].set_offsets(pos); art[1].set_offsets(pos[mine])
                art[1].set_facecolor(row["snaps"][k]["patch_colours"][img])
            elif p != "input":
                s = row["snaps"][k]
                art.set_data(s["explanation"][img] if p == "explanation" else s[p][img].astype(float))
        marker.set_xdata([t, t]); head.set_text(f"{title}training step {snaps0[k]['step']}")
        return [marker, head]

    return animation.FuncAnimation(fig, frame, frames=(n - 1) * frames_between + 1, interval=120, blit=False)


def show_checkpoint(rows, image, panels, k):
    """A still of checkpoint k (for the slider)."""
    fig, axes = plt.subplots(len(rows), len(panels), figsize=(3.0 * len(panels), 3.0 * len(rows)), squeeze=False)
    for r, row in enumerate(rows):
        img = row.get("image", image)
        for j, p in enumerate(panels):
            ax = axes[r, j]; ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(PANEL_TITLES[p], fontsize=9)
            if j == 0:
                ax.set_ylabel(row["label"], fontsize=9)
            if p == "umap":
                mine = row["cloud"].of(img)
                ax.scatter(*row["emb"][k].T, c="0.82", s=3, linewidths=0)
                ax.scatter(*row["emb"][k][mine].T, c=row["snaps"][k]["patch_colours"][img], s=14,
                           edgecolors="black", linewidths=0.3)
                ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
            else:
                draw(ax, p, row, k, img)
    fig.suptitle(f"training step {rows[0]['snaps'][k]['step']}", fontsize=11)
    plt.show()

PANELS = ("input", "umap", "explanation", "uncertainty", "saliency")   # every panel, always


def correlation(proportions, labels, n_classes):
    """(K, C) Pearson correlation between concept proportions and the class indicators."""
    Y = (labels[:, None] == np.arange(n_classes)[None]).astype(float)
    P = proportions - proportions.mean(0); Y = Y - Y.mean(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        R = (P.T @ Y) / (np.sqrt((P ** 2).sum(0))[:, None] * np.sqrt((Y ** 2).sum(0))[None])
    return np.nan_to_num(R)


In [ ]:
from torchvision import datasets, transforms

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
SIZE = 160
augment = transforms.Compose([transforms.RandomResizedCrop(SIZE, scale=(0.35, 1)), transforms.RandomHorizontalFlip(),
                              transforms.ToTensor()])
plain = transforms.Compose([transforms.Resize(SIZE), transforms.CenterCrop(SIZE), transforms.ToTensor()])
normalise = transforms.Normalize(MEAN, STD)
train_raw = datasets.Imagenette("data", split="train", size="160px", download=True)   # about 100 MB, once
val_raw = datasets.Imagenette("data", split="val", size="160px", download=False)
class_names = [c[0] if isinstance(c, (tuple, list)) else c for c in train_raw.classes]
train_targets = np.array([y for _, y in getattr(train_raw, "_samples", [])]) if hasattr(train_raw, "_samples") else \
    np.array([train_raw[i][1] for i in range(len(train_raw))])
val_targets = np.array([y for _, y in getattr(val_raw, "_samples", [])]) if hasattr(val_raw, "_samples") else \
    np.array([val_raw[i][1] for i in range(len(val_raw))])


class Images(torch.utils.data.Dataset):
    def __init__(self, raw, idx, tf):
        self.raw, self.idx, self.tf = raw, np.asarray(idx), tf
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        img, y = self.raw[int(self.idx[i])]
        return self.tf(img), y

rng = np.random.default_rng(SEED)
keep = np.concatenate([np.nonzero(train_targets != RARE_CLASS)[0],
                       rng.choice(np.nonzero(train_targets == RARE_CLASS)[0],
                                  int(RARE_SHARE * np.sum(train_targets == RARE_CLASS)), replace=False)])
loader = torch.utils.data.DataLoader(Images(train_raw, keep, augment), batch_size=128, shuffle=True, num_workers=4,
                                     drop_last=True)
probe_idx = np.concatenate([rng.choice(np.nonzero(val_targets == c)[0], PROBE_PER_CLASS, replace=False)
                            for c in range(len(class_names))])
probe = torch.stack([plain(val_raw[int(i)][0]) for i in probe_idx]); probe_labels = val_targets[probe_idx]
probe_images = [to_uint8(x) for x in probe]
SHOW = [c * PROBE_PER_CLASS for c in range(len(class_names))]         # one image per class
IMAGE = RARE_CLASS
print(np.bincount(train_targets[keep]), "training images per class")

In [ ]:
import torchvision
from nave_xai import NAVEImage
from nave_xai.models import TorchBackbone
def nave_snapshot(backbone, fit_images, show_images, layers, cloud_holder):
    """At a checkpoint: NAVE fitted on fit_images; for every showable image its explanation (labels),
    uncertainty and NAVE saliency; and the patch cloud's composite features."""
    nave = NAVEImage(backbone, layers=layers, k=K, seed=SEED).fit(fit_images)
    exps = nave.predict(show_images, uncertainty=True)
    try:
        saliency = [nave.saliency(im).astype(np.float16) for im in show_images]
    except ValueError:                                          # a model without classifier head
        saliency = [np.zeros(im.shape[:2], np.float16) for im in show_images]
    out = dict(labels=[e.explanation for e in exps], uncertainty=[e.uncertainty.astype(np.float16) for e in exps],
               saliency=saliency)
    feats, grid, cell = composite_patches(backbone, cloud_holder["images"], layers)
    if "cloud" not in cloud_holder:
        cloud_holder["cloud"] = PatchCloud(len(cloud_holder["images"]), grid[0] * grid[1], cloud_holder["show"],
                                           N_GRAY, SEED)
        cloud_holder["centres"] = cell_centres(cloud_holder["images"][0].shape[0], cell)
    out["cloud"] = cloud_holder["cloud"].gather(feats)
    return out

model = torchvision.models.resnet34(num_classes=len(class_names)).to(DEVICE)
optimiser = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, max_lr=0.1, total_steps=EPOCHS * len(loader))
backbone = TorchBackbone(model, mean=MEAN, std=STD, device=DEVICE)
holder = dict(images=probe_images, show=SHOW)
C = len(class_names)
snapshots = []

@torch.no_grad()
def snapshot(step):
    model.eval()
    preds = torch.cat([model(normalise(b).to(DEVICE)).argmax(1).cpu() for b in probe.split(100)]).numpy()
    nave = NAVEImage(backbone, layers=LAYERS, k=K, seed=SEED).fit(probe_images)
    props = np.stack([np.bincount(e.explanation.ravel(), minlength=K) / e.explanation.size for e in nave.predict(probe_images)])
    R = correlation(props, probe_labels, C)
    owner = np.where(R.max(1) > 0.1, R.argmax(1), -1)
    s = nave_snapshot(backbone, probe_images[::2], [probe_images[i] for i in SHOW], LAYERS, holder)
    snapshots.append(dict(step=step, concepts=np.bincount(owner[owner >= 0], minlength=C),
                          per_class=np.array([(preds[probe_labels == c] == c).mean() for c in range(C)]), **s))
    model.train()
    print(f"step {step:6d}: concepts per class {snapshots[-1]['concepts']}")

step = 0
snapshot(step)
for epoch in range(EPOCHS):
    for x, y in loader:
        loss = torch.nn.functional.cross_entropy(model(normalise(x).to(DEVICE)), y.to(DEVICE))
        optimiser.zero_grad(); loss.backward(); optimiser.step(); schedule.step()
        step += 1
        if epoch == 0 and step in (25, 50, 100, 200):
            snapshot(step)
    snapshot(step)
colour_snapshots(snapshots, K, holder["centres"])

## Concepts and accuracy per class

In [ ]:
def animate_bars(frames_between=6):
    fig, (a_c, a_a) = plt.subplots(1, 2, figsize=(11, 3.6), dpi=60)
    x = np.arange(C); colours = plt.get_cmap("tab10")(x % 10)
    bars = a_c.bar(x, snapshots[0]["concepts"], color=colours); a_c.set_ylim(0, max(s["concepts"].max() for s in snapshots) + 1)
    accs = a_a.bar(x, snapshots[0]["per_class"], color=colours); a_a.set_ylim(0, 1)
    for a, t in ((a_c, "concepts per class"), (a_a, "accuracy per class")):
        a.set_title(t, fontsize=9); a.set_xticks(x, [n[:8] for n in class_names], rotation=60, fontsize=6)
        a.get_xticklabels()[RARE_CLASS].set_fontweight("bold")
    head = fig.suptitle("", fontsize=11); n = len(snapshots)

    def frame(f):
        k = int(round(f / frames_between))
        for b, v in zip(bars, snapshots[k]["concepts"]):
            b.set_height(v)
        for b, v in zip(accs, snapshots[k]["per_class"]):
            b.set_height(v)
        head.set_text(f"'{class_names[RARE_CLASS]}' {int(1 / RARE_SHARE)} x rarer  |  step {snapshots[k]['step']}")
        return [head, *bars, *accs]

    plt.rcParams["animation.embed_limit"] = 100
    return animation.FuncAnimation(fig, frame, frames=(n - 1) * frames_between + 1, interval=120, blit=False)

anim = animate_bars()
display(HTML(anim.to_jshtml()))
anim.save("imbalance_bars.gif", writer=animation.PillowWriter(fps=8), dpi=70)
plt.close()

## The animation

UMAP of the patches, each checkpoint kept close to the previous one (`AlignedUMAP`): gray, patches of the other
probe images; in colour, the shown image's patches, by their concept. Change `IMAGE` and re-run only the cells
from here.

In [ ]:
emb_previous = {}
rows = [dict(label="ResNet34", snaps=snapshots, cloud=holder["cloud"], inputs=[probe_images[i] for i in SHOW],
             emb=aligned_umap([s["cloud"] for s in snapshots], UMAP_EPOCHS))]
curves = [("rare class accuracy", [s["per_class"][RARE_CLASS] for s in snapshots]),
          ("other classes' accuracy", [np.delete(s["per_class"], RARE_CLASS).mean() for s in snapshots])]
anim = animate_training(rows, curves, IMAGE, PANELS, title="imbalance  |  ")
display(HTML(anim.to_jshtml()))
anim.save("imbalance.gif", writer=animation.PillowWriter(fps=8), dpi=70)
plt.close()

## With a slider

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(lambda checkpoint, image: show_checkpoint(rows, image, PANELS, checkpoint),
                     checkpoint=widgets.IntSlider(len(rows[0]["snaps"]) - 1, 0, len(rows[0]["snaps"]) - 1, 1),
                     image=widgets.IntSlider(IMAGE, 0, len(SHOW) - 1, 1));
except ImportError:
    show_checkpoint(rows, IMAGE, PANELS, len(rows[0]["snaps"]) - 1)

## Things to look at

- how many concepts the rare class gets, and when;
- `IMAGE` (one image per class): the rare class's patches among the others in the UMAP;
- `RARE_SHARE`, or class weights in the loss to compensate.

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```